# Week 1: Foundations — Cleaning & Exploring the Applicant Dataset

**Name:** Shiza Ali  
**Internship Program:** NextGen Learners Data Science Internship Program

## Objective:
The objective of this task is to learn how to clean, organize, and explore an applicant dataset using Python and Pandas. I will identify and handle missing values, remove duplicate records, standardize text, correct data types, and prepare a clean dataset for further analysis.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('applicants.csv')

In [3]:
df.shape          # how many rows and columns? 

(80, 7)

In [4]:
df.head()         # preview the first 5 rows 

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Ali Farooq,ali.farooq1@example.com,036-494-4897,APP DEV,Government College University Faisalabad,2026-08-08,SELECTED
1,Saad Farooq,saad.farooq2@example.com,037-807-5619,Web Development,COMSATS University,28/08/2026,Under review
2,Noor Ali,noor.ali3@example.com,034-840-5908,graphics design,The University of Faisalabad,09-11-2026,Under Review
3,LAIBA SIDDIQUI,laiba.siddiqui4@example.com,032-668-2487,digital marketing,The University of Faisalabad,2026-10-12,Under review
4,Iqra Farooq,iqra.farooq5@example.com,037-275-4275,GRAPHIC DESIGN,COMSATS University,12/08/2026,UNDER REVIEW


In [5]:
df.info()         # data types + non-null counts per column 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    78 non-null     object
 1   Email             76 non-null     object
 2   Phone             80 non-null     object
 3   Domain Applied    80 non-null     object
 4   University        77 non-null     object
 5   Application Date  80 non-null     object
 6   Status            77 non-null     object
dtypes: object(7)
memory usage: 4.5+ KB


In [6]:
df.isnull().sum() # exact count of missing values per column 

Applicant Name      2
Email               4
Phone               0
Domain Applied      0
University          3
Application Date    0
Status              3
dtype: int64

In [7]:
df.duplicated().sum()  # how many exact duplicate rows exist

np.int64(2)

## Initial Data Exploration — Observations

The dataset contains **80 rows and 7 columns**, providing information about applicants, including their names, email addresses, phone numbers, applied domains, universities, application dates, and application status. During the initial exploration, I checked the dataset's structure, missing values, data types, and duplicate records using Pandas.

The missing-value analysis shows that **Applicant Name has 2 missing values, Email has 4 missing values, University has 3 missing values, and Status has 3 missing values**. The Phone, Domain Applied, and Application Date columns have no missing values at this stage. Email has the highest number of missing values, which may make it difficult to identify applicants or check for duplicate records.

The dataset also contains **2 exact duplicate rows**, which may affect the accuracy of the analysis if they are not removed. Therefore, I will remove exact duplicates and check for near-duplicate records by standardizing email addresses.

Another important observation is that the **Application Date column is currently stored as an object (text) data type instead of an actual datetime format**. I will convert it into a proper datetime format to make date-related analysis easier. I will also review the text columns for inconsistent formatting and handle missing values carefully.

Overall, the initial exploration shows that the dataset requires some cleaning before it can be used for reliable analysis. The next steps will include handling missing values, removing duplicate records, standardizing text values, converting application dates, and checking the quality of the cleaned dataset.

 ## Handle Missing Values (Column by Column, With  Justification) 

In [10]:
df = df.dropna(subset=['Applicant Name'])  

**Following the removal of records with missing applicant names, 78 rows remain.**

In [11]:
df['Email'] = df['Email'].fillna('not_provided@example.com')

With a compact dataset size of 78 rows and 7 columns, dropping rows with missing email addresses can be counterproductive. 

#### Why Using a Placeholder?
* **Preserving Sample Size:** In a small dataset, every record is valuable. Discarding rows directly reduces statistical power and can introduce bias into your analysis or predictive models.
* **Retaining Multidimensional Data:** Keeping the row ensures that the applicant's remaining attributes (such as scores, categories, or timestamps) remain available for exploratory data analysis, visualizations, or feature engineering.
* **Preventing Runtime Errors:** Replacing null values with a structured string like `"not_provided@example.com"` prevents unexpected `NullType` exceptions during string processing or regex validation steps.

In [12]:
df['University'] = df['University'].fillna('Not Specified') 

Unlike missing applicant names or contact details, missing university information is less critical and does not warrant removing entire records from the dataset. 
#### Reasoning:
* **Data Retention:** Retaining rows with missing university data ensures that other critical variables in the record remain available for downstream analysis.
* **Consistency & Clarity:** Filling missing entries with an explicit placeholder string (`"Not Specified"`) maintains data consistency and ensures categorical fields can be grouped cleanly without encountering `NaN` handling issues.

In [13]:
df['Status'] = df['Status'].fillna('Under Review')

To ensure complete categorical tracking across all records, missing entry statuses were imputed using a logical default state.
#### Reasoning:
* **Business Logic:** An unpopulated status typically indicates that an application is still in progress rather than rejected or finalized. Assigning `'Under Review'` accurately reflects the pending nature of these records.
* **Data Integrity:** Eliminating missing values in key categorical columns ensures seamless downstream operations, such as status-based grouping, filtering, and summary metrics.

## Removing Duplicate Applicants 

In [16]:
df = df.drop_duplicates(subset=['Email'], keep='first')

In [17]:
df.shape

(73, 7)

### Why We Chose Email Over Applicant's Name for Duplicate Detection?

After dropping duplicates based on the `Email` column, your dataset was successfully reduced from **78 rows to 73 rows** (removing 5 duplicate submissions). Choosing email as the unique identifier rather than the applicant's name comes down to reliability and data accuracy:

* **Names are Not Unique:** Multiple applicants can share the exact same name (e.g., common first and last name combinations). Relying on names alone risks accidentally deleting legitimate, distinct applicants.
* **Resilience to Variations:** Applicants often alter how they type their name across submissions—such as using a nickname, omitting a middle name, or changing capitalization. An email address remains consistent.
* **Behavioral Reality:** When someone submits an application twice (often by accident or due to a submission error), they almost always use the same email address to track it, making `Email` the most dependable fingerprint for finding true duplicate submissions.

## Standardize Text Fields

In [18]:
# Check original domain values
print(df['Domain Applied'].unique())

['APP DEV' 'Web Development' 'graphics design' 'digital marketing'
 'GRAPHIC DESIGN' 'DataScience' 'Data Science' 'Mobile App Development'
 'webdev' 'Graphic Design' 'web dev' ' WEB DEV ' 'Digital marketting'
 'WEB DEV' 'DIGITAL MARKETING' 'graphic design' 'data science'
 'Data science ' 'App Development' 'DATA SCIENCE' 'Web development'
 'appdev' 'Digital Marketing']


In [19]:
# Step 1: Remove extra spaces and convert to lowercase
df['Domain Applied'] = (
    df['Domain Applied']
    .str.strip()
    .str.lower()
)

In [20]:
# Step 2: Create a mapping dictionary
domain_mapping = {
    # Web Development
    'app dev': 'Mobile App Development',
    'appdev': 'Mobile App Development',
    'app development': 'Mobile App Development',
    'mobile app development': 'Mobile App Development',

    # Web Development
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',

    # Graphic Design
    'graphics design': 'Graphic Design',
    'graphic design': 'Graphic Design',

    # Data Science
    'datascience': 'Data Science',
    'data science': 'Data Science',

    # Digital Marketing
    'digital marketing': 'Digital Marketing',
    'digital marketting': 'Digital Marketing'
}

In [21]:
# Step 3: Apply mapping
df['Domain Applied'] = df['Domain Applied'].map(domain_mapping)

In [22]:
# Step 4: Display unique standardized values
print("Unique domain values after standardization:")
print(df['Domain Applied'].unique())

Unique domain values after standardization:
['Mobile App Development' 'Web Development' 'Graphic Design'
 'Digital Marketing' 'Data Science']


In [23]:
# Step 5: Check missing values
print("\nMissing values after standardization:")
print(df['Domain Applied'].isnull().sum())


Missing values after standardization:
0


In [24]:
# Step 6: Count each domain
print("\nDomain counts:")
print(df['Domain Applied'].value_counts())


Domain counts:
Domain Applied
Web Development           18
Graphic Design            16
Digital Marketing         15
Mobile App Development    12
Data Science              12
Name: count, dtype: int64


### Standardizing the Domain Applied Column

The Domain Applied column contained inconsistent text values representing the same domains. For example, "web dev", "webdev", and "WEB DEV" referred to Web Development, while "graphics design" and "GRAPHIC DESIGN" represented Graphic Design. Similarly, "DataScience" and "Data Science" referred to the same domain. Variations were also present in Mobile App Development and Digital Marketing, including the misspelled value "Digital marketting".

To solve this problem, I first removed extra spaces using `str.strip()` and converted all domain values to lowercase using `str.lower()`. Then, I created a mapping dictionary to convert the different spellings and formats into five consistent domain labels: "Web Development", "Graphic Design", "Data Science", "Mobile App Development", and "Digital Marketing".

After applying the mapping, I checked the unique values in the Domain Applied column to verify that the standardization was successful. This process helps prevent the same domain from being counted as multiple categories and makes the dataset more consistent and reliable for further analysis.

In [25]:
#Standardizing the Status column
df['Status'] = df['Status'].str.strip().str.title()

In [26]:
# Display unique status values
print("Unique status values after standardization:")
print(df['Status'].unique())

Unique status values after standardization:
['Selected' 'Under Review' 'Rejected']


In [27]:
# Count missing values
print("\nMissing values in Status:")
print(df['Status'].isnull().sum())


Missing values in Status:
0


In [28]:
# Count applicants in each status category
print("\nStatus counts:")
print(df['Status'].value_counts())


Status counts:
Status
Under Review    34
Selected        23
Rejected        16
Name: count, dtype: int64


### Handling Missing Values and Standardizing the Status Column

The `Status` column contained inconsistent capitalization, with values such as "SELECTED", "Selected", and "selected". Similarly, "UNDER REVIEW", "Under review", and "under review" represented the same category, while "REJECTED" and "rejected" also differed in capitalization.

During the initial exploration, I identified blank or missing values in the `Status` column and imputed them with `"Under Review"` as a default pending state. 

After handling the missing values, I used the `str.strip()` and `str.title()` methods to remove extra spaces and standardize the capitalization. The final normalized categories are **"Selected"**, **"Under Review"**, and **"Rejected"**. Finally, I checked the unique values, category counts, and missing-value count to verify that the cleaning process was successful.

## Fix Data Types (Especially Dates and Phone Numbers)

**Convert Application Date to a real date type:**

In [29]:
# Remove extra spaces from date values
date_text = df['Application Date'].astype('string').str.strip()

In [30]:
# Treat ######## and blank values as missing
date_text = date_text.replace(r'^#+$', pd.NA, regex=True)
date_text = date_text.replace('', pd.NA)

In [31]:
# Convert dates into datetime format
df['Application Date'] = pd.to_datetime(
    date_text,
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [32]:
#Check results
print("First 10 converted dates:")
print(df['Application Date'].head(10))

First 10 converted dates:
0   2026-08-08
1   2026-08-28
2   2026-11-09
3   2026-10-12
4   2026-08-12
5   2026-05-09
6   2026-09-02
7   2026-08-30
8   2026-09-29
9          NaT
Name: Application Date, dtype: datetime64[ns]


In [33]:
print("\nMissing or invalid dates after conversion:")
print(df['Application Date'].isnull().sum())


Missing or invalid dates after conversion:
3


In [34]:
print("\nData type:")
print(df['Application Date'].dtype)


Data type:
datetime64[ns]


In [36]:
# Filling missing dates instead of dropping rows using forward fill (.ffill())
df['Application Date'] = df['Application Date'].ffill()

In [38]:
print("Missing dates after handling:")
print(df['Application Date'].isnull().sum())

Missing dates after handling:
0


In [39]:
print("Dataset shape after filling missing dates:")
print(df.shape)

Dataset shape after filling missing dates:
(73, 7)


### Application Date Parsing, Cleaning, and Imputation

Date values often contain hidden formatting issues such as leading/trailing whitespaces, Excel overflow artifacts (e.g., `########`), or empty strings. To ensure the date field is accurate and ready for time-series analysis, we perform string standardization before converting to `datetime64` and handling any invalid or missing entries.

#### 1. Rationale & Cleaning Steps
* **Whitespace & Noise Stripping:** Removes accidental spaces and handles corrupted Excel display strings (`########`) by replacing them with `pd.NA`.
* **Flexible Datetime Conversion:** Uses `format='mixed'` along with `dayfirst=True` to parse non-uniform date formats safely while converting unparseable/corrupted strings into `NaT` via `errors='coerce'`.
* **Why Use Forward Fill (`.ffill()`) Instead of Dropping Rows?** 
  * In a compact dataset, dropping rows due to missing dates would unnecessarily discard valuable applicant information.
  * We use **Forward Fill (`.ffill()`)** to propagate the last valid chronological date forward to fill the missing `NaT` values. This preserves our total sample size while keeping the time-series continuity intact without introducing arbitrary or out-of-range dates.

**Clean up Phone numbers:**

In [48]:
# Strip leading/trailing whitespace, then remove hyphens and internal spaces
df['Phone'] = (
    df['Phone']
    .astype(str)
    .str.strip()
    .str.replace('-', '', regex=False)
    .str.replace(' ', '', regex=False)
)

In [49]:
# Verify the cleaned phone numbers
print("Cleaned Phone Numbers Sample:")
print(df['Phone'].head())

Cleaned Phone Numbers Sample:
0    364944897
1    378075619
2    348405908
3    326682487
4    372754275
Name: Phone, dtype: object


In [50]:
print("\nMissing or invalid dates after conversion:")
print(df['Phone'].isnull().sum())


Missing or invalid dates after conversion:
0


Data stored as plain text cannot be properly sorted, filtered, or analyzed mathematically. To ensure technical integrity across the dataset, numerical and date attributes were converted into their appropriate data types.
#### Cleaning & Standardizing Phone Numbers
Phone numbers contain formatting noise (spaces and hyphens) and must be cleaned while keeping them as **string (`object`) types** to preserve leading zeros (e.g., `0364944897`).

* **Rationale:** Converting phone numbers to numeric types (`int64`) strips leading zeros, corrupting contact data.

In [51]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 73 entries, 0 to 77
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    73 non-null     object        
 1   Email             73 non-null     object        
 2   Phone             73 non-null     object        
 3   Domain Applied    73 non-null     object        
 4   University        73 non-null     object        
 5   Application Date  73 non-null     datetime64[ns]
 6   Status            73 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 4.6+ KB


## Writing the Data Quality Summary 